Expire Changed Rows from Day 4


In [0]:
-- Step 1: Expire rows that have changed in Bronze (Day 4 modifications)
UPDATE nestle_dev_silver.core.sales_transactions_scd2 AS target
SET dbt_valid_to = current_date(),
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM nestle_dev_bronze.sql_server.sales_transactions AS src
    WHERE src.transaction_id = target.transaction_id
      AND (src.amount <> target.amount 
           OR src.quantity <> target.quantity
           OR src.channel <> target.channel)
  );

SELECT COUNT(*) as expired_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = FALSE;

 Insert New Current Rows from Day 4 Bronze

In [0]:
-- Step 2: Insert new current rows (Day 4 additions + updated rows)
INSERT INTO nestle_dev_silver.core.sales_transactions_scd2
SELECT 
  src.transaction_id,
  src.product_id,
  src.region,
  src.channel,
  src.customer_id,
  src.quantity,
  src.unit_price,
  src.amount,
  CAST(current_date() AS DATE) as dbt_valid_from,
  CAST('2099-12-31' AS DATE) as dbt_valid_to,
  TRUE as dbt_is_current
FROM nestle_dev_bronze.sql_server.sales_transactions src
WHERE NOT EXISTS (
  SELECT 1 FROM nestle_dev_silver.core.sales_transactions_scd2 tgt
  WHERE tgt.transaction_id = src.transaction_id
    AND tgt.dbt_is_current = TRUE
);

SELECT COUNT(*) as new_current_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE;

Verify Complete SCD2 History

In [0]:
SELECT 
  COUNT(*) as total_scd2_rows,
  COUNT(CASE WHEN dbt_is_current = TRUE THEN 1 END) as current_rows,
  COUNT(CASE WHEN dbt_is_current = FALSE THEN 1 END) as expired_rows,
  MIN(dbt_valid_from) as earliest_valid_from,
  MAX(dbt_valid_to) as latest_valid_to
FROM nestle_dev_silver.core.sales_transactions_scd2;

Sample SCD2 Record with History

In [0]:
SELECT 
  transaction_id,
  product_id,
  amount,
  dbt_valid_from,
  dbt_valid_to,
  dbt_is_current,
  ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY dbt_valid_from) as version
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE transaction_id IN (SELECT transaction_id FROM nestle_dev_silver.core.sales_transactions_scd2 WHERE dbt_is_current = FALSE LIMIT 1)
ORDER BY transaction_id, dbt_valid_from;